In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp

# Load raw CSV
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/ ev_charging_data ")

# Check it loaded correctly
print(f"Total records: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(5)

Total records: 242417
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state_province: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- ports: string (nullable = true)
 |-- power_kw: double (nullable = true)
 |-- power_class: string (nullable = true)
 |-- is_fast_dc: string (nullable = true)

+------+--------------------+------------+--------------+------------+---------+---------+-----+--------+------------------+----------+
|    id|                name|        city|state_province|country_code| latitude|longitude|ports|power_kw|       power_class|is_fast_dc|
+------+--------------------+------------+--------------+------------+---------+---------+-----+--------+------------------+----------+
|307660|    Av. de Tarragona|     Andorra|       UNKNOWN|          AD|42.505254| 1.528861|   10|   300.0|DC_ULTRA_(>=150kW)|

In [0]:
from pyspark.sql.functions import col, when

# Drop nulls in critical columns
df_clean = df_raw.dropna(subset=["id", "power_kw", "power_class"])

# Safely cast — replace UNKNOWN/bad values with None
def safe_double(c):
    return when(col(c).rlike(r'^-?\d+(\.\d+)?$'), col(c).cast("double")).otherwise(None)

def safe_int(c):
    return when(col(c).rlike(r'^\d+$'), col(c).cast("integer")).otherwise(None)

df_clean = df_clean \
    .withColumn("power_kw", safe_double("power_kw")) \
    .withColumn("ports", safe_int("ports")) \
    .withColumn("latitude", safe_double("latitude")) \
    .withColumn("longitude", safe_double("longitude")) \
    .withColumn("is_fast_dc", when(col("is_fast_dc") == "True", True).otherwise(False))

# Drop rows where lat/lon are null (were UNKNOWN)
df_clean = df_clean.filter(
    col("latitude").isNotNull() & col("longitude").isNotNull()
)

print(f"Records after cleaning: {df_clean.count()}")
df_clean.printSchema()

Records after cleaning: 237751
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state_province: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ports: integer (nullable = true)
 |-- power_kw: double (nullable = true)
 |-- power_class: string (nullable = true)
 |-- is_fast_dc: boolean (nullable = false)



In [0]:
delta_path = "/Volumes/workspace/default/ ev_charging_data /delta_table"

df_clean.write.format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print(f"Data saved to Delta Lake at: {delta_path}")

Data saved to Delta Lake at: /Volumes/workspace/default/ ev_charging_data /delta_table


In [0]:
df_delta = spark.read.format("delta").load("/Volumes/workspace/default/ ev_charging_data /delta_table")

print(f"Delta table record count: {df_delta.count()}")
df_delta.show(5)

Delta table record count: 237751
+------+--------------------+------------+--------------+------------+---------+---------+-----+--------+------------------+----------+
|    id|                name|        city|state_province|country_code| latitude|longitude|ports|power_kw|       power_class|is_fast_dc|
+------+--------------------+------------+--------------+------------+---------+---------+-----+--------+------------------+----------+
|307660|    Av. de Tarragona|     Andorra|       UNKNOWN|          AD|42.505254| 1.528861|   10|   300.0|DC_ULTRA_(>=150kW)|      true|
|301207|Parquing Costa Ro...|      Encamp|       UNKNOWN|          AD|42.537213| 1.727014|   10|    22.0| AC_HIGH_(22-49kW)|     false|
|301206|         Hotel Naudi|Unknown City|       UNKNOWN|          AD|42.576811| 1.666061|    1|    11.0|  AC_L2_(7.5-21kW)|     false|
|301205|Hotel Piolets Sol...|Unknown City|       UNKNOWN|          AD|42.576466| 1.667317|    1|    22.0| AC_HIGH_(22-49kW)|     false|
|301204|       

In [0]:
spark.sql("DROP TABLE IF EXISTS default.ev_charging")

spark.sql("""
    CREATE TABLE IF NOT EXISTS default.ev_charging
    USING DELTA
    AS SELECT * FROM delta.`/Volumes/workspace/default/ ev_charging_data /delta_table`
""")

print("Table registered successfully!")
spark.sql("SELECT * FROM default.ev_charging LIMIT 5").show()

Table registered successfully!
+------+--------------------+------------+--------------+------------+---------+---------+-----+--------+------------------+----------+
|    id|                name|        city|state_province|country_code| latitude|longitude|ports|power_kw|       power_class|is_fast_dc|
+------+--------------------+------------+--------------+------------+---------+---------+-----+--------+------------------+----------+
|307660|    Av. de Tarragona|     Andorra|       UNKNOWN|          AD|42.505254| 1.528861|   10|   300.0|DC_ULTRA_(>=150kW)|      true|
|301207|Parquing Costa Ro...|      Encamp|       UNKNOWN|          AD|42.537213| 1.727014|   10|    22.0| AC_HIGH_(22-49kW)|     false|
|301206|         Hotel Naudi|Unknown City|       UNKNOWN|          AD|42.576811| 1.666061|    1|    11.0|  AC_L2_(7.5-21kW)|     false|
|301205|Hotel Piolets Sol...|Unknown City|       UNKNOWN|          AD|42.576466| 1.667317|    1|    22.0| AC_HIGH_(22-49kW)|     false|
|301204|        H